# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Hussainhhgh/flyrank-ml-internship/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [10]:
!git clone https://github.com/Hussainhhgh/flyrank-ml-internship.git 2>/dev/null
import pandas as pd
df = pd.read_csv('flyrank-ml-internship/data/raw/content_refresh_anonymized.csv')
print(df.columns.tolist())
print(df.shape)
df.head(3)

['content_id', 'client_id', 'search_volume', 'competition', 'competition_level', 'cpc', 'content_type', 'main_intent', 'word_count', 'char_count', 'provider_used', 'model_used', 'impressions_90d', 'clicks_90d', 'pageviews_90d', 'sessions_90d', 'users_90d', 'engaged_sessions_90d', 'ai_sessions_90d', 'scroll_events_90d', 'days_with_impressions', 'days_with_sessions', 'impressions_last_30d', 'clicks_last_30d', 'sessions_last_30d', 'impressions_prev_30d', 'clicks_prev_30d', 'sessions_prev_30d', 'content_age_days', 'age_tier', 'age_tier_order', 'days_since_last_update', 'freshness_tier', 'word_count_tier', 'char_count_tier', 'ctr', 'avg_position', 'engagement_rate', 'scroll_rate', 'ai_traffic_pct', 'impression_tier', 'position_tier', 'trend_direction', 'trend_pct']
(30000, 44)


,content_id,client_id,search_volume,competition,competition_level,cpc,content_type,main_intent,word_count,char_count,...,char_count_tier,ctr,avg_position,engagement_rate,scroll_rate,ai_traffic_pct,impression_tier,position_tier,trend_direction,trend_pct
0,content_304f48230142,client_f369cb89fc,10.0,0.67,HIGH,2.05,keyword article,transactional,3221.0,20457.0,...,15000-25000,0.76,10.6,5.88,4.55,0.0,good,striking,down,-41.4
1,content_a1fb4e703a9e,client_4e07408562,90.0,0.01,LOW,0.05,keyword article,informational,2481.0,15562.0,...,15000-25000,0.05,20.3,0.00,10.00,0.0,good,page_3_5,down,-57.7
2,content_9aa793d4d895,client_7f2253d7e2,0.0,0.00,LOW,0.00,keyword article,informational,3515.0,23643.0,...,15000-25000,0.09,36.5,0.00,28.57,0.0,good,page_3_5,down,-60.9


## 1. My rule and its reason codes

**My rule:** Flag pages that are stale (low freshness) AND underperforming on CTR relative to their position as needing a refresh. A page that is fresh and performing fine needs no action.

**Reason codes:** STALE, CTR_UNDERPERFORM, STALE_CTR_UNDERPERFORM, NONE

**Signal 1 (staleness, behind refresh flags):** checked freshness_tier against decline rate. **Signal 2 (CTR vs position, behind CTR-fix logic):** checked ctr against position_tier.

**Verdict — Signal 1 (freshness_tier): MIXED.** Decline rate rises from 0-30 days (51.1%) to 31-90 days (58.9%) to 91-180 days (61.1%), fitting the staleness story — but reverses at 181+ days (47.1%). That oldest bucket is much smaller (n=174 vs n=20,480 for 0-30), so it may be too small to trust. Staleness looks directionally real in the first three buckets but breaks down at the tail.

**Verdict — Signal 2 (position_tier): OPPOSITE** of the simple expectation. Decline rate did not rise as position worsened — instead top_3 (24.1%) and deep (34.4%) showed the LOWEST decline rates, while striking (61.0%), page_1 (57.0%), and page_3_5 (56.2%), the middle positions, showed the highest. Decline risk appears concentrated in the middle of the ranking pack, not at the extremes. Position alone, in the simple "worse position = more risk" direction, does not hold — a useful negative result.

In [11]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

print(df['freshness_tier'].unique())
print(df['position_tier'].unique())

print("\nSignal 1: freshness_tier")
bucket1 = df.groupby('freshness_tier').agg(
    n=('content_id', 'count'),
    avg_ctr=('ctr', 'mean'),
    pct_declining=('trend_direction', lambda x: (x == 'down').mean())
).reset_index()
print(bucket1)

print("\nSignal 2: position_tier")
bucket2 = df.groupby('position_tier').agg(
    n=('content_id', 'count'),
    avg_ctr=('ctr', 'mean'),
    pct_declining=('trend_direction', lambda x: (x == 'down').mean())
).reset_index()
print(bucket2)


['0-30' '91-180' '181+' '31-90']
['striking' 'page_3_5' 'page_1' 'top_3' 'deep']

Signal 1: freshness_tier
  freshness_tier      n   avg_ctr  pct_declining
0           0-30  20480  0.609021       0.511377
1           181+    174  3.693276       0.471264
2          31-90    175  0.117543       0.588571
3         91-180   9171  0.238367       0.611057

Signal 2: position_tier
  position_tier      n   avg_ctr  pct_declining
0          deep   1319  0.150212       0.344200
1        page_1  11814  0.652467       0.569663
2      page_3_5   7242  0.222484       0.561585
3      striking   7304  0.323239       0.609529
4         top_3   2321  1.483611       0.240844


## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

**Rule encoded:** Score +2 for staleness (freshness_tier in 31-90 or 91-180 days, per Signal 1's MIXED-but-directional finding), +2 for mid-pack position (position_tier in striking/page_1/page_3_5, per Signal 2's OPPOSITE finding that risk concentrates in the middle, not the extremes). Score ≥3 → refresh_priority, ≥1 → monitor, else no_action.

In [12]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

import os

def score_row(row):
    score = 0
    reason = []

    # Staleness: 31-90 and 91-180 tiers showed highest decline risk
    if row['freshness_tier'] in ['31-90', '91-180']:
        score += 2
        reason.append('STALE_MIDAGE')

    # Position: middle-pack positions showed highest decline risk (not worst position)
    if row['position_tier'] in ['striking', 'page_1', 'page_3_5']:
        score += 2
        reason.append('MIDPACK_POSITION')

    reason_code = '_'.join(reason) if reason else 'NONE'
    return pd.Series([score, reason_code])

df[['action_score', 'reason_code']] = df.apply(score_row, axis=1)

def action_label(score):
    if score >= 3:
        return 'refresh_priority'
    elif score >= 1:
        return 'monitor'
    else:
        return 'no_action'

df['action_label'] = df['action_score'].apply(action_label)

queue = df.sort_values('action_score', ascending=False)[
    ['content_id', 'action_score', 'reason_code', 'action_label']
]

os.makedirs('work/outputs', exist_ok=True)
queue.to_csv('work/outputs/baseline_action_score.csv', index=False)
print(queue.head(20))
print(f"\nAction label counts:\n{queue['action_label'].value_counts()}")


                 content_id  action_score                    reason_code  \
29984  content_a6568a7c07d7             4  STALE_MIDAGE_MIDPACK_POSITION   
14     content_91067a14431a             4  STALE_MIDAGE_MIDPACK_POSITION   
13     content_a5a2fbc76336             4  STALE_MIDAGE_MIDPACK_POSITION   
12     content_42fb2cad9ecf             4  STALE_MIDAGE_MIDPACK_POSITION   
9      content_c27558df2b0c             4  STALE_MIDAGE_MIDPACK_POSITION   
27     content_7ea135180dd9             4  STALE_MIDAGE_MIDPACK_POSITION   
24     content_0e23e310d404             4  STALE_MIDAGE_MIDPACK_POSITION   
16     content_78bd1d4a1d4d             4  STALE_MIDAGE_MIDPACK_POSITION   
46     content_c59d46264834             4  STALE_MIDAGE_MIDPACK_POSITION   
26930  content_2a25085efa41             4  STALE_MIDAGE_MIDPACK_POSITION   
26928  content_dcba430284bf             4  STALE_MIDAGE_MIDPACK_POSITION   
26957  content_61cce40fd25a             4  STALE_MIDAGE_MIDPACK_POSITION   
26953  conte

## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

Top-20 review — action, reason code, confidence note, and what would make it wrong.

1. content_c27558df2b0c — refresh_priority (STALE_MIDAGE_MIDPACK_POSITION). 91-180 days old, page_1, ctr=0.16, avg_position=4.9. Confidence: MEDIUM — age signal present, CTR not clearly broken. Would be wrong if: this page's CTR is already near-typical for position 4.9 — the age signal may be doing all the work here.

2. content_42fb2cad9ecf — refresh_priority (STALE_MIDAGE_MIDPACK_POSITION). 91-180 days, page_1, ctr=1.76 (high), avg_position=5.6. Confidence: LOW — CTR looks strong, likely false positive. Would be wrong if: a 1.76 CTR at position 5.6 is actually strong performance — flagging this looks like a false positive driven by age alone.

3. content_a5a2fbc76336 — refresh_priority (STALE_MIDAGE_MIDPACK_POSITION). 91-180 days, page_3_5, ctr=0.00, avg_position=39.8. Confidence: MEDIUM — deep position may need more than a refresh. Would be wrong if: position 39.8 is so deep that no realistic refresh would move CTR — this may need a content/keyword rethink, not a refresh.

4. content_91067a14431a — refresh_priority (STALE_MIDAGE_MIDPACK_POSITION). 91-180 days, page_3_5, ctr=0.00, avg_position=27.1. Confidence: HIGH — near-zero CTR with weak position is a clear risk signal. Would be wrong if: zero CTR at this depth is normal/expected for the tier broadly, not a decay signal specific to this page.

5. content_78bd1d4a1d4d — refresh_priority (STALE_MIDAGE_MIDPACK_POSITION). 91-180 days, page_1, ctr=0.15, avg_position=8.9. Confidence: MEDIUM — CTR modest but not alarming. Would be wrong if: 0.15 CTR is within normal range for position ~9 — nothing unusual is happening here.

6. content_0e23e310d404 — refresh_priority (STALE_MIDAGE_MIDPACK_POSITION). 91-180 days, page_1, ctr=0.27, avg_position=4.1. Confidence: LOW — decent CTR at a strong position, likely false positive. Would be wrong if: ctr=0.27 at position 4.1 suggests this page is actually performing fine; refreshing it may waste a slot.

7. content_7ea135180dd9 — refresh_priority (STALE_MIDAGE_MIDPACK_POSITION). 91-180 days, page_1, ctr=0.17, avg_position=8.1, oldest in this batch at 445 days. Confidence: MEDIUM — very old, but CTR not clearly broken. Would be wrong if: this page has been stable at this CTR for a long time — age alone doesn't mean it's actively declining.

8. content_c59d46264834 — refresh_priority (STALE_MIDAGE_MIDPACK_POSITION). 91-180 days, page_1, ctr=0.08 (low), avg_position=6.1. Confidence: MEDIUM-HIGH — low CTR at a decent position is a real signal. Would be wrong if: a CTR this low at a decent position is a snippet/title problem, not something a content refresh would fix — reason code should arguably be CTR_FIX, not refresh.

9. content_845a243716ba — refresh_priority (STALE_MIDAGE_MIDPACK_POSITION). 91-180 days, page_3_5, ctr=0.09, avg_position=26.6. Confidence: MEDIUM — low CTR but may be typical for this tier. Would be wrong if: this combination is just typical for page_3_5 depth broadly, not specific to this page.

10. content_654f4b8bf560 — refresh_priority (STALE_MIDAGE_MIDPACK_POSITION). 91-180 days, striking, ctr=0.00, avg_position=15.8. Confidence: HIGH — zero CTR despite a striking snippet is unusual and worth flagging. Would be wrong if: zero CTR despite a striking-snippet position suggests a technical/indexing issue, not a content-quality issue a refresh would fix.

11. content_50096ce97c93 — refresh_priority (STALE_MIDAGE_MIDPACK_POSITION). 91-180 days, striking, ctr=0.32, avg_position=11.6. Confidence: LOW — solid CTR for a striking snippet, likely false positive. Would be wrong if: 0.32 CTR at position 11.6 with a striking snippet is arguably solid — may not need prioritization.

12. content_dcba430284bf — refresh_priority (STALE_MIDAGE_MIDPACK_POSITION). 91-180 days, striking, ctr=0.17, avg_position=14.5. Confidence: MEDIUM — average performer for its tier. Would be wrong if: this looks like a typical performer, not an outlier needing urgent action.

13. content_2a25085efa41 — refresh_priority (STALE_MIDAGE_MIDPACK_POSITION). 91-180 days, page_1, ctr=0.16, avg_position=5.5. Confidence: MEDIUM — similar to #1, CTR not clearly broken for this position. Would be wrong if: CTR is already near-typical for this position — may be a false positive from age alone.

14. content_37106924f264 — refresh_priority (STALE_MIDAGE_MIDPACK_POSITION). 91-180 days, page_1, ctr=1.01 (high), avg_position=3.4. Confidence: LOW — strong CTR at a top position, likely false positive. Would be wrong if: ctr=1.01 at position 3.4 is a strong performer — flagging this is likely a false positive from age alone.

15. content_13612df1afe8 — refresh_priority (STALE_MIDAGE_MIDPACK_POSITION). 91-180 days, page_1, ctr=0.16, avg_position=7.7. Confidence: MEDIUM — within normal range, age may be overriding a fine CTR signal. Would be wrong if: this CTR is normal for the position; age alone shouldn't drive the flag.

16. content_e342ea5fbafc — refresh_priority (STALE_MIDAGE_MIDPACK_POSITION). 91-180 days, page_3_5, ctr=0.04, avg_position=28.5. Confidence: MEDIUM — low CTR but may be standard for this depth. Would be wrong if: this depth/CTR combo is standard for page_3_5 broadly — nothing page-specific is going wrong.

17. content_5d35ae9a76c0 — refresh_priority (STALE_MIDAGE_MIDPACK_POSITION). 91-180 days, striking, ctr=0.73 (high), avg_position=14.2. Confidence: LOW — notably strong CTR, likely false positive. Would be wrong if: ctr=0.73 is a strong CTR — this looks like a false positive driven by age alone.

18. content_6ed4f9b27daf — refresh_priority (STALE_MIDAGE_MIDPACK_POSITION). 91-180 days, striking, ctr=0.00, avg_position=12.7. Confidence: HIGH — zero CTR with a striking snippet is a clear anomaly. Would be wrong if: zero CTR with a striking snippet signals a technical issue (e.g. broken snippet rendering), not something a content refresh addresses.

19. content_61cce40fd25a — refresh_priority (STALE_MIDAGE_MIDPACK_POSITION). 91-180 days, striking, ctr=0.31, avg_position=16.8, oldest at 362 days. Confidence: LOW-MEDIUM — moderate CTR, may just be stable rather than declining. Would be wrong if: a 362-day-old page holding a 0.31 CTR may just be stable, not declining.

20. content_a6568a7c07d7 — refresh_priority (STALE_MIDAGE_MIDPACK_POSITION). 91-180 days, striking, ctr=0.21, avg_position=12.4. Confidence: MEDIUM — average performer for its tier and position. Would be wrong if: no clear signal it's actively worsening vs. just aging.

In [13]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

top20 = df.merge(queue.head(20)[['content_id']], on='content_id')[
    ['content_id', 'freshness_tier', 'position_tier', 'ctr', 'avg_position',
     'content_age_days', 'action_score', 'reason_code', 'action_label']
]
print(top20.to_string())


              content_id freshness_tier position_tier   ctr  avg_position  content_age_days  action_score                    reason_code      action_label
0   content_c27558df2b0c         91-180        page_1  0.16           4.9               257             4  STALE_MIDAGE_MIDPACK_POSITION  refresh_priority
1   content_42fb2cad9ecf         91-180        page_1  1.76           5.6               124             4  STALE_MIDAGE_MIDPACK_POSITION  refresh_priority
2   content_a5a2fbc76336         91-180      page_3_5  0.00          39.8               238             4  STALE_MIDAGE_MIDPACK_POSITION  refresh_priority
3   content_91067a14431a         91-180      page_3_5  0.00          27.1               124             4  STALE_MIDAGE_MIDPACK_POSITION  refresh_priority
4   content_78bd1d4a1d4d         91-180        page_1  0.15           8.9               307             4  STALE_MIDAGE_MIDPACK_POSITION  refresh_priority
5   content_0e23e310d404         91-180        page_1  0.27           

## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

**Weak picks:** Five rows (content_42fb2cad9ecf, content_0e23e310d404, content_50096ce97c93, content_37106924f264, content_5d35ae9a76c0) were flagged refresh_priority despite having notably high CTR (0.27–1.76) — well above what the low-CTR rows in this same batch show. These look like false positives: the rule is currently driven entirely by freshness_tier + position_tier, with no CTR sanity check, so a page can be "stale" and "mid-pack position" while still performing great, and still get flagged. A stronger v2 rule would only flag pages where CTR is ALSO below some threshold, not just old + mid-pack.

**Leakage check:** The rule uses only freshness_tier and position_tier as scoring inputs — both are properties knowable at decision time, before any outcome is known. trend_direction and trend_pct (the actual decline label/outcome) were explicitly excluded from the scoring logic and were only used earlier, in Section 1, to sanity-check whether the signals correlated with real outcomes — never as inputs to the score itself. No future-window or label-derived data was used to build the rule.

In [14]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

# Weak picks: LOW-confidence rows from the top 20 (high CTR flagged anyway)
weak_ids = ['content_42fb2cad9ecf', 'content_0e23e310d404', 'content_50096ce97c93',
            'content_37106924f264', 'content_5d35ae9a76c0']
weak_picks = df[df['content_id'].isin(weak_ids)][
    ['content_id', 'freshness_tier', 'position_tier', 'ctr', 'avg_position', 'action_score']
]
print("Weak picks (high CTR flagged despite it):")
print(weak_picks)

# Leakage check: confirm trend_direction / trend_pct were NOT used as inputs to the score
print("\nColumns used in scoring:")
print("freshness_tier, position_tier — both available at decision time, no future data")
print("\nConfirming trend_direction/trend_pct excluded from rule inputs:")
scoring_inputs = ['freshness_tier', 'position_tier']
leaked = [c for c in scoring_inputs if c in ['trend_direction', 'trend_pct']]
print(f"Leaked label columns in scoring inputs: {leaked if leaked else 'NONE — confirmed clean'}")


Weak picks (high CTR flagged despite it):
                 content_id freshness_tier position_tier   ctr  avg_position  \
12     content_42fb2cad9ecf         91-180        page_1  1.76           5.6   
24     content_0e23e310d404         91-180        page_1  0.27           4.1   
26914  content_50096ce97c93         91-180      striking  0.32          11.6   
26935  content_37106924f264         91-180        page_1  1.01           3.4   
26949  content_5d35ae9a76c0         91-180      striking  0.73          14.2   

       action_score  
12                4  
24                4  
26914             4  
26935             4  
26949             4  

Columns used in scoring:
freshness_tier, position_tier — both available at decision time, no future data

Confirming trend_direction/trend_pct excluded from rule inputs:
Leaked label columns in scoring inputs: NONE — confirmed clean


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.